# B1W2

**Chapters 02** &nbsp;·&nbsp; Block 1, Week 2 &nbsp;·&nbsp; lab Thu 20 Aug 2026

**Fill this notebook in and submit it.** Restart the kernel and run all before you
submit: the saved output must be the output of one clean pass.

Do not rename the functions or change their arguments or return values.

---


# B1W2 — Block 1, Week 2 — Reading a design matrix

> **45 minutes · 20 marks · lab desktop, no AI, no internet.**
> Chapter 02 · Notes: *Chapter02* · Bank: *Chapter02_Questions* Section C
> Format and data contract: *AIS5102_Coding_Quizzes* · Key: *Chapter02_CodingQuiz_Key*

You are given a thousand Raman spectra and told nothing about the samples. By the end of this
paper you will have found — without being told they exist — that the thousand spectra fall into
**three kinds of material**, and you will be able to say what physically distinguishes them.

Every step is a few lines of code. The work is in knowing *why* each line is there, so each step
below gives you the mathematics first and the code second. Read the formula, then write the line
that performs it.


## Notation

Fix this now and the rest of the paper reads easily.

| Symbol | Meaning | Shape |
|---|---|---|
| $N$ | number of spectra (rows, *measurements*) | 1000 |
| $W$ | number of wavenumber bins (columns, *features*) | 1024 |
| $\mathbf X = X_{nk}$ | the design matrix — spectrum $n$, bin $k$ | $(N, W)$ |
| $\mu_k$ | mean of feature $k$ | $(W,)$ |
| $\sigma_k$ | standard deviation of feature $k$ | $(W,)$ |
| $\mathbf X_c$ | mean-centred design matrix | $(N, W)$ |
| $\mathbf S$ | scatter matrix | $(W, W)$ |
| $\boldsymbol\Sigma$ | covariance matrix | $(W, W)$ |
| $\mathbf C$ | correlation matrix | $(W, W)$ |
| $\lambda_k, \mathbf v_k$ | eigenvalue and eigenvector of $\boldsymbol\Sigma$ | scalar, $(W,)$ |
| $\mathbf P$ | projection onto the leading eigenvectors | $(N, K)$ |

**Index convention.** $n$ always runs over spectra, $k$ and $i,j$ always over wavenumber bins.
Whenever you write `axis=0` you are summing over $n$; `axis=1` sums over $k$. Getting this wrong
is the single most common way to lose marks in this paper.


## What to submit

Fill in `B1W2_template.ipynb`, open on your desktop, which already contains every stub below, one
function to a cell. **Restart the kernel and run all before you submit** — the saved output must be
the output of one clean pass, or the marks that depend on running your code are lost.

**Do not rename the functions or change their arguments or return values.** Inside the body, write
whatever helpers you like.

**Every figure needs labelled axes with units**, a title saying what it shows, and a legend if more
than one thing is drawn. A figure with bare axes earns nothing.

---


## Step 1 — Load the design matrix and the labels *(with step 2: 2 marks)*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def q021(path="b1_data1.npz"):
    """Load the sitting's data.

    Returns
    -------
    X : ndarray, shape (N, W)
        One Raman spectrum per row. Rows are measurements, columns are features.
    axis : ndarray, shape (W,)
        The label of each column: the wavenumber that column was measured at,
        in cm^-1. Ascending.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


`axis` is the **feature label vector**. It is not data — you never do arithmetic on `X` with it.
It is what turns a column index into a physical quantity, and you will need it at almost every
step from here on. Keep it beside `X` at all times.

**Report** — `X.shape`, `axis.shape`, and the first and last wavenumber.

---


## Step 2 — Look at your data *(marked with step 1)*

**Before any arithmetic, look at the matrix.** This is not a formality. Nearly every disaster in
applied machine learning is visible in a plot of the raw design matrix and invisible in its
summary statistics.

```python
plt.imshow(X, aspect="auto", cmap="bone")
```

You will find this nearly useless as written, and that is the lesson. The intensities span
492 to 2666, but almost all of that range is used by **one** very bright band. Everything else is
crushed into the bottom few percent of the colour scale and you see a dark rectangle with a single
bright stripe.

**Fix the dynamic range.** Two standard tools, either is fine:

$$
\text{gamma:}\quad X' = X^{\gamma},\ \gamma \approx 0.3
\qquad\qquad
\text{log:}\quad X' = \log_{10} X
$$

In matplotlib you do not have to transform the array — pass a *norm* instead, which keeps the
colourbar honest about the original units:

```python
from matplotlib.colors import LogNorm, PowerNorm
plt.imshow(X, aspect="auto", cmap="bone", norm=PowerNorm(gamma=0.3))
```

Use `aspect="auto"`, not `"equal"` — the matrix is 1000 × 1024, and a square aspect on a wide
figure wastes most of the canvas.

**Plot** the design matrix twice, side by side: once with linear intensity and once with your
gamma or log scaling. Label the $x$ axis *feature (wavenumber bin)*, the $y$ axis *measurement
(spectrum)*, and give the colourbar the units *intensity, counts*.

**Interpret** — one line: with the scaling fixed, you can see vertical stripes at a handful of
column positions, and the stripes are not equally bright in every row. Say what a vertical stripe
means physically, and what it means that its brightness varies down the column.

---


## Step 3 — Feature statistics: the first two cumulants *(2 marks)*

For each column $k$ compute its mean and its spread across the thousand measurements. These are
the first two **cumulants** of that feature's distribution:

$$
\kappa_1 = \mu_k = \frac{1}{N}\sum_{n=1}^{N} X_{nk}
\qquad\qquad
\kappa_2 = \sigma_k^2 = \frac{1}{N-1}\sum_{n=1}^{N}\bigl(X_{nk}-\mu_k\bigr)^2
$$

Both are sums **over $n$**, so both are `axis=0`, and both come out with shape $(W,)$ — one number
per feature, not one per spectrum. If you get shape $(N,)$ you have summed over the wrong index.


In [ ]:
def q023(X):
    """First and second cumulant of every feature.

    Parameters
    ----------
    X : ndarray, shape (N, W)

    Returns
    -------
    mu : ndarray, shape (W,)
        Mean of each column.
    sd : ndarray, shape (W,)
        Standard deviation of each column. Use ddof=1, the unbiased estimate,
        so that it agrees with the covariance matrix you build in step 6.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**Plot** $\mu_k$ and $\sigma_k$ against `axis` — *against the wavenumber, not the bin index* — in
two stacked panels sharing an $x$ axis. Label them separately; they are not in the same units and
must not share a $y$ axis.

**Report** — the range of $\mu$, the range of $\sigma$, and the wavenumber at which each is
largest.

**Interpret** — two lines. First: the two curves peak in the same place. Say what that tells you
about where the *information* in this dataset lives, and why a feature with $\sigma_k$ near zero
cannot help you tell one spectrum from another no matter how large its $\mu_k$ is. Second: the
$\sigma$ curve is spiky — nearly flat, with a few sharp excursions. A spike in $\sigma$ is either
a real band whose strength varies between samples, or a faulty detector column. Say how you would
tell those two apart using only what is on this plot.

---


## Step 4 — Mean-centre, and check it *(1 mark)*

Subtract each feature's own mean from that feature:

$$
(X_c)_{nk} \;=\; X_{nk} - \mu_k
$$

Every step after this one — scatter, covariance, correlation, eigenvectors — is a statement about
$\mathbf X_c$, not $\mathbf X$. Centring is what makes "how do features *vary together*" a
meaningful question; without it every covariance is dominated by the fact that all the intensities
are large positive numbers.


In [ ]:
def q024(X, mu):
    """Subtract the feature means.

    Returns
    -------
    Xc : ndarray, shape (N, W)
        X with every column's mean removed, so that Xc.mean(axis=0) is zero
        to within floating-point round-off.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


`mu` has shape $(W,)$ and `X` has shape $(N, W)$. NumPy broadcasting aligns them from the right,
so `X - mu` already does the correct thing. Writing `X - mu[None, :]` makes the intent explicit
and costs nothing; writing `X - mu[:, None]` raises an error, which is broadcasting protecting you.

**Report** — `np.abs(Xc.mean(axis=0)).max()`.

**Interpret** — one line: the number is not exactly zero. Say what it is, roughly how large you
expect it to be for `float64` data of this magnitude, and why testing `Xc.mean(axis=0) == 0` would
be the wrong check to write.

---


## Step 5 — Name the bands *(2 marks)*

Until now the columns have been anonymous. A Raman spectrum has named features, and five of them
matter here. **You may take these positions as given.**

| Band | cm⁻¹ | What it reports |
|---|---|---|
| a-Si | 490 | amorphous silicon — from the **substrate**, not the graphene |
| Si-O | 978 | silicon oxide — also substrate |
| D | 1351 | **defects** — this mode is forbidden in perfect graphene and is activated by disorder |
| G | 1580 | sp² carbon — in-plane stretching of the carbon lattice, always present |
| 2D | 2701 | double resonance — reports **how many layers** there are and how they stack |

The three carbon bands are the ones that say something about the graphene. Spectroscopists rarely
quote them raw; they quote **ratios**, because a ratio cancels anything that scales the whole
spectrum — laser power, integration time, focus, how much material happened to sit under the beam.
$I_D/I_G$ is the standard defect metric and $I_{2D}/I_G$ the standard layer metric.

Build a dictionary of band centres and mark them on your figures:


In [ ]:
bands = {"a-Si": 490, "Si-O": 978, "D": 1351, "G": 1580, "2D": 2701}


`plt.vlines(x, ymin, ymax)` draws vertical lines at every position in `x`. Use it to drop a marker
at each band centre across the full height of the axes, and `ax.annotate(name, (centre, y))` to
label each one. A neat idiom for "full height, whatever the data does":

```python
ax.vlines(list(bands.values()), *ax.get_ylim(), color="0.7", lw=1, zorder=0)
```

`zorder=0` puts the lines behind the data. Call it *after* you have plotted, so that `get_ylim()`
returns the range the data actually occupies.

**Plot** — redraw step 3's two panels, now with all five bands marked and named.

**Interpret** — one line: not every spike in the $\sigma$ curve sits on a named band, and not every
named band shows a spike. Say which bands carry real variation across your thousand measurements,
and which of the five you would expect to be nearly constant given that every spectrum was measured
on the same kind of wafer.

---


## Step 6 — Scatter, covariance, correlation *(4 marks)*

Three matrices, each built from the last. All three are $(W, W)$ — feature against feature.

**Scatter.** Sum over measurements of the product of two centred features:

$$
S_{ij} \;=\; \sum_{n=1}^{N} (X_c)_{ni}\,(X_c)_{nj}
\qquad\Longleftrightarrow\qquad
\mathbf S \;=\; \mathbf X_c^{\mathsf T}\mathbf X_c
$$

That equivalence is worth staring at. The double sum on the left *is* a matrix product; the `@`
operator performs it, and you should never write the loop.

**Covariance.** Divide by $N-1$ to make it an average rather than a total:

$$
\Sigma_{ij} \;=\; \frac{S_{ij}}{N-1}
$$

The $N-1$ rather than $N$ is Bessel's correction: you spent one degree of freedom estimating
$\mu_k$ in step 3. This is also why step 3 asked for `ddof=1` — so that $\Sigma_{kk} = \sigma_k^2$
exactly.

**Correlation.** Divide out the scale of each feature:

$$
C_{ij} \;=\; \frac{\Sigma_{ij}}{\sigma_i\,\sigma_j}
\qquad\Longleftrightarrow\qquad
\mathbf C \;=\; \mathbf D^{-1}\boldsymbol\Sigma\,\mathbf D^{-1},\quad
\mathbf D = \operatorname{diag}(\sigma)
$$

In code the division is by the **outer product** $\sigma\sigma^{\mathsf T}$, `np.outer(sd, sd)`,
which has shape $(W, W)$ — not by `sd` once. Dividing once gives a matrix that is not symmetric and
whose diagonal is not 1, and it is the most common error on this task.


In [ ]:
def q026(Xc, sd):
    """Scatter, covariance and correlation of a centred design matrix.

    Returns
    -------
    S : ndarray, shape (W, W)
        Scatter, Xc.T @ Xc.
    Sigma : ndarray, shape (W, W)
        Covariance, S / (N - 1).
    Corr : ndarray, shape (W, W)
        Correlation. Unit diagonal, every entry in [-1, 1]. Build it from
        Sigma and the outer product of sd. Do not call a library correlation
        function.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**Plot** all three as images in a **1 × 3 panel, each with its own colourbar**. They are in wildly
different units — that is the point, and a shared colourbar would hide it. A diverging colormap
centred on zero (`cmap="RdBu_r"` with `vmin=-m, vmax=+m`) reads far better than a sequential one,
because the sign of a covariance is meaningful.

**Report** — the shape of each, and $\operatorname{Tr}(\boldsymbol\Sigma)$. That trace is the
**total variance** of the dataset, $\sum_k \sigma_k^2$, and you will use it again in step 8.

**Interpret** — two lines. First: the covariance and correlation images look completely different.
Say which one you can read a relationship off, and what the other one is dominated by. Second: the
correlation matrix has visible bright blocks away from the diagonal. Say what a bright off-diagonal
block at $(i,j)$ means when $i$ and $j$ are hundreds of cm⁻¹ apart.


### Step 6a — Which features move together? *(optional)*

Rank the off-diagonal entries of $\mathbf C$ by $|C_{ij}|$ and report the strongest pairs, giving
each as a **pair of wavenumbers with band names attached** — not as a pair of column indices.

One wrinkle makes this more interesting than it looks: neighbouring columns are trivially
correlated, because a Raman band is many bins wide and adjacent bins measure the same peak. Those
pairs are real but uninformative. Exclude pairs closer than about 150 cm⁻¹ and report what survives.

**Interpret** — the strongest surviving pair is two *different* bands. Say why those two in
particular should rise and fall together, and what that tells you about the samples.

---


## Step 7 — Eigenvectors of the covariance, computed stably *(2 marks)*

We now ask: along which directions in the 1024-dimensional feature space does the data actually
extend? Those directions are the eigenvectors of $\boldsymbol\Sigma$:

$$
\boldsymbol\Sigma\,\mathbf v_k \;=\; \lambda_k\,\mathbf v_k
\qquad\Longleftrightarrow\qquad
\boldsymbol\Sigma \;=\; \mathbf V\boldsymbol\Lambda\mathbf V^{\mathsf T}
$$

with $\mathbf V$ orthogonal ($\mathbf V^{\mathsf T}\mathbf V = \mathbf I$) and $\boldsymbol\Lambda$
diagonal. This is *Chapter02*'s **rotate, stretch, rotate**:
$\mathbf V^{\mathsf T}$ rotates the cloud so its natural axes lie along the coordinate axes,
$\boldsymbol\Lambda$ says how far it extends along each, and $\mathbf V$ rotates back.

**Use `numpy.linalg.eigh`, not `numpy.linalg.eig`.** This matters, and is worth understanding
rather than memorising:

| | `eig` | `eigh` |
|---|---|---|
| Assumes | any square matrix | **symmetric** (Hermitian) |
| Returns | `complex128`, in no guaranteed order | `float64`, **ascending** |
| On this matrix | imaginary parts around $10^{-13}$ — pure round-off, but they make the array complex and poison everything downstream | exact real values |

A covariance matrix is symmetric by construction, so its eigenvalues are provably real. `eig` does
not know that, so it solves the general problem and hands back round-off as imaginary parts. Using
`eig` here and then reading `eigval[0]` as the largest happens to give the right answer on this
file — and that is luck, not correctness, because `eig` never promised an order.

`eigh` **does** promise an order: ascending. You want descending, so reverse it — and reverse the
eigenvector *columns* to match, or you will pair the right eigenvalues with the wrong directions.


In [ ]:
def q027(Sigma):
    """Eigendecomposition of a covariance matrix, largest first.

    Returns
    -------
    lam : ndarray, shape (W,)
        Eigenvalues in DESCENDING order. Return them exactly as the
        decomposition gives them -- do not clip negatives or take absolute
        values.
    V : ndarray, shape (W, W)
        Eigenvectors as COLUMNS, in the same order as lam, so that V[:, k]
        is the direction belonging to lam[k].
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**Report** — the three largest eigenvalues, and a check that $\mathbf V$ is orthonormal: the
largest absolute deviation of $\mathbf V^{\mathsf T}\mathbf V$ from the identity.

**Interpret** — one line: some of the smallest eigenvalues come back slightly **negative**, which
is impossible for a covariance matrix. Say what they really are, and why the task tells you not to
clip them.

---


## Step 8 — The eigenvalue spectrum: how many directions? *(2 marks)*

Plot $\lambda_k$ against $k$ on a **logarithmic $y$ axis**. On a linear axis you see one enormous
spike and a flat line; the structure of this dataset spans several orders of magnitude, and a log
axis is the only way to see it.

**A trap you will hit.** Step 7 told you not to clip the negative eigenvalues, and $\log$ of a
negative number is undefined — matplotlib will silently drop those points. Either plot
$|\lambda_k|$, or plot only the leading few dozen. Say in a comment which you chose and why.

Now quantify "how many directions matter". The trace of the covariance is the total variance, and
is also the sum of the eigenvalues:

$$
\operatorname{Tr}\boldsymbol\Sigma \;=\; \sum_{k} \Sigma_{kk} \;=\; \sum_k \lambda_k
$$

so the fraction of the dataset's total variance captured by the leading $K$ directions is

$$
f_K \;=\; \frac{\sum_{k=1}^{K}\lambda_k}{\sum_{k=1}^{W}\lambda_k}
$$


In [ ]:
def q028(lam, target=0.95):
    """How many leading directions are needed.

    Parameters
    ----------
    lam : ndarray, shape (W,)
        Eigenvalues, descending.
    target : float
        Fraction of the total variance to reach.

    Returns
    -------
    frac : ndarray, shape (W,)
        Cumulative fraction of total variance: frac[k-1] is f_K for K = k.
    K : int
        The smallest K with frac[K-1] >= target.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**Plot** the eigenvalue spectrum on a log $y$ axis, and the cumulative fraction $f_K$ against $K$ in
a second panel, with a horizontal line at your target.

**Report** — $f_1$, $f_2$, $f_3$, $f_5$ as percentages, and $K$ at the 95% target.

**Interpret** — two lines. First: one eigenvalue is far larger than all the others. Using what you
found in step 3 about where $\sigma$ is largest, say which physical feature of the samples that
direction describes — and note that it is **not** a property of the graphene. Second: the data
nominally lives in 1024 dimensions. Say how many it *actually* occupies, and what that means about
how much of your design matrix was doing useful work.

---


## Step 9 — What does an eigenvector look like? *(1 mark)*

Each $\mathbf v_k$ has shape $(W,)$ — the same shape as a spectrum. **So plot it as one**, against
`axis`, with the bands marked as in step 5.

This is the step that makes the abstraction concrete. An eigenvector of the covariance is not an
abstract direction; it is a *pattern of co-variation* — a recipe saying "when this sample differs
from the average, these bins go up together and those go down together."

**Plot** $\mathbf v_0$, $\mathbf v_1$ and $\mathbf v_2$ against wavenumber, offset vertically so
they do not overlap (add a constant to each), with the five bands marked.

**Report** — for each of the three, the wavenumbers of the five largest $|v_k|$ entries, with band
names.

**Interpret** — one line: these three vectors are mutually orthogonal,
$\mathbf v_i^{\mathsf T}\mathbf v_j = \delta_{ij}$, yet they all put weight on overlapping sets of
bands. Say how both can be true at once — what orthogonality does and does not forbid.

---


## Step 10 — Project the spectra, and look at the cloud *(2 marks)*

Each row of $\mathbf X_c$ is a point in 1024 dimensions. Step 8 said only a handful of directions
matter, so give each spectrum its coordinates **along those directions**:

$$
P_{nd} \;=\; \sum_{k=1}^{W} (X_c)_{nk}\,V_{kd}
\qquad\Longleftrightarrow\qquad
\mathbf P \;=\; \mathbf X_c\,\mathbf V_{:,1:K}
$$

$\mathbf P$ has shape $(N, K)$: still one row per spectrum, but $K$ numbers instead of 1024. Nothing
has been fitted and nothing has been learned — this is a change of coordinates, a rotation of the
same cloud.


In [ ]:
def q0210(Xc, V, n_components=3):
    """Project each spectrum onto the leading eigenvectors.

    Returns
    -------
    P : ndarray, shape (N, n_components)
        Row n holds the coordinates of spectrum n along V[:, 0], V[:, 1], ...
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**A property worth checking**, because it tells you the projection is right:

$$
\operatorname{Var}\bigl(P_{:,d}\bigr) \;=\; \lambda_d
$$

The variance of the $d$-th projected coordinate *is* the $d$-th eigenvalue. If yours does not
match, the likely causes are that you projected the uncentred $\mathbf X$, or that you took
`V[:, :K]` without reversing to descending order in step 7.

**Plot a pair plot** of the first three coordinates: a 3 × 3 grid where panel $(i,j)$ scatters
coordinate $i$ against coordinate $j$, with a histogram of each coordinate down the diagonal. Every
panel needs axis labels; sharing $x$ per column and $y$ per row keeps it readable.

**Report** — `P.shape`, and $\operatorname{Var}(P_{:,d})$ against $\lambda_d$ for $d = 0, 1, 2$.

**Interpret** — one line: the cloud is not a single blob. Say how many groups you can see, in which
panel of the pair plot they separate most clearly, and whether any *pair* of coordinates separates
them better than the first coordinate alone.

---


## Step 11 — What are the groups? *(2 marks)*

Split the spectra into **three approximate types** using a threshold on the coordinate you named in
step 10. Your split does not have to match anyone else's — the cloud has two dense lobes with a thin
bridge between them, so where exactly you cut the bridge is a judgement call.

```python
cut = np.quantile(P[:, 0], [0.34, 0.72])     # a starting point, not the answer
types = np.digitize(P[:, 0], cut)
```

Now bring back the feature labels. For each type, average the **raw** spectra belonging to it:

$$
m^{(c)}_k \;=\; \frac{1}{|c|}\sum_{n \in c} X_{nk}
$$

One average spectrum per type, each of shape $(W,)$ — and now `axis` earns its keep, because a
class-average spectrum plotted against wavenumber, with the bands marked, tells you what the classes
*are*.


In [ ]:
def q0211(X, types, axis, bands, halfwidth=80.0):
    """Class-average spectra and the two diagnostic band ratios.

    Parameters
    ----------
    types : ndarray of int, shape (N,)
        Which type each spectrum was assigned to, values 0..T-1.
    halfwidth : float
        A band's intensity is the mean of the columns within this many
        cm^-1 of the band centre.

    Returns
    -------
    means : ndarray, shape (T, W)
        The average raw spectrum of each type.
    ratios : ndarray, shape (T, 2)
        Column 0 is the mean I_D / I_G of that type, column 1 the mean
        I_2D / I_G.
    """
    return


In [ ]:
# --- your work for this task ---
# Call the function above on the real data, make the requested plot,
# and print every requested number with the label the paper gives it.


**Plot** the three class-average spectra on shared axes with the five bands marked, and a second
figure zooming on the D–G–2D region where they differ.

**Report** — the number of spectra in each type, and the table of $I_D/I_G$ and $I_{2D}/I_G$ per
type.

**Interpret** — two lines. First: using the band table in step 5, say what physically distinguishes
the three types — a higher $D/G$ means more of one thing, say what. Second: the two ratios move in
**opposite** directions across the types. Say why that is one coherent story about the material
rather than two separate findings.

---


## Optional — go further if you have time *(unmarked)*

The natural next questions. None is required, none earns marks, all three return in later chapters.

**O1 · Rotate before you split.** Your threshold in step 11 was a vertical cut in the pair plot, but
the groups may be separated along a *tilted* line. Apply a 2-D rotation to the first two
coordinates,

$$
\mathbf R(\theta)=\begin{pmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{pmatrix},
\qquad \mathbf P' = \mathbf P_{:,0:2}\,\mathbf R(\theta)
$$

sweep $\theta$, and find the angle at which a single threshold on the first rotated coordinate
separates the groups best. Say what "best" means in your answer — you have to choose a criterion,
and choosing it is the exercise. *(Chapter 14 answers this properly, with clustering.)*

**O2 · One feature instead of a projection.** Compute a single number per spectrum — the mean
intensity within ±80 cm⁻¹ of the G band — and histogram it. Compare its structure against the first
projected coordinate. If one engineered feature recovers the groups that a 1024-dimensional
eigendecomposition found, say what the eigendecomposition bought you, and what it cost.
*(Feature engineering is the whole of *B1W3* Part B.)*

**O3 · How much of a spectrum survives $K$ directions?** Reconstruct the design matrix from the
leading $K$ eigenvectors,

$$
\widehat{\mathbf X} \;=\; \mathbf P\,\mathbf V_{:,1:K}^{\mathsf T} + \boldsymbol\mu
$$

and plot one original spectrum against its reconstruction for $K = 1, 2, 5, 20$. Report the relative
error $\lVert\mathbf X - \widehat{\mathbf X}\rVert_F / \lVert\mathbf X\rVert_F$ against $K$, and
compare it against $1 - f_K$ from step 8. *(This is the reconstruction loss of Chapter 08.)*

---


## Marks

| Step | | Marks |
|---|---|---|
| 1 · 2 | Load, and look at the raw matrix | 2 |
| 3 | Feature statistics — the first two cumulants | 2 |
| 4 | Mean-centre, and check it | 1 |
| 5 | Name the bands, and mark them | 2 |
| 6 | Scatter, covariance, correlation | 4 |
| 7 | Eigendecomposition, computed stably | 2 |
| 8 | The eigenvalue spectrum — how many directions? | 2 |
| 9 | Eigenvectors, plotted as spectra | 1 |
| 10 | Project, and look at the cloud | 2 |
| 11 | What are the groups? | 2 |
| | **Total** | **20** |

Within each step: **half** for correct, runnable code; **a quarter** for the requested figures and
numbers; **a quarter** for the interpretation.

Eleven steps in forty-five minutes is about four minutes each. They are short — but they are
strictly sequential, so **if a step defeats you, hard-code a plausible result and move on.** A later
step done well on an imperfect input earns full marks; a blank cell earns nothing.


## What you carry forward

`Xc`, the band dictionary, and the **three sample types** from step 11.
*Week 3* opens by rebuilding them, and asks how tight each type is and how far apart they
are. It also asks you to do in one line what step 11 did in ten — by engineering the right feature
instead of projecting.
